# ShallowLandslider output analysis

This notebook analyses the self-describing run directories produced by either `ShallowLandslider_quickstart.ipynb` or the v1.2 YAML CLI. It does not rerun the landslide model. Run the quick-start notebook first for a small runout-enabled example, then run this notebook from top to bottom. Large raster outputs remain closed until explicitly requested.

Expected layout:

```text
runs/
  <run-id>/
    manifest.json
    summary.json
    regions.csv (and optionally regions.parquet)
    rasters.zarr/ (or rasters/*.npy)
```

## 1. Imports and paths

Run this notebook from the repository root, or change `RUNS_ROOT` to the directory immediately containing the individual run folders.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis import discover_runs, load_region_ensemble, load_run, plot_run

RUNS_ROOT = Path("runs")
ANALYSIS_OUTPUT = Path("analysis_output")
SELECTED_ONLY = True

ANALYSIS_OUTPUT.mkdir(parents=True, exist_ok=True)

## 2. Discover and summarise runs

Discovery is deliberately one directory deep: `RUNS_ROOT/<run-id>/manifest.json`. No raster arrays are opened in this section. The summary is written as both CSV for dataframe/spreadsheet use and indented JSON for quick inspection.

In [ ]:
run_dirs = discover_runs(RUNS_ROOT)
if not run_dirs:
    raise FileNotFoundError(
        f"No run directories found below {RUNS_ROOT.resolve()}. "
        "Set RUNS_ROOT to the directory containing the run folders."
    )

print(f"Found {len(run_dirs)} run(s)")
for run_dir in run_dirs[:10]:
    print(run_dir)
if len(run_dirs) > 10:
    print(f"... and {len(run_dirs) - 10} more")

In [ ]:
run_summaries = []
for run_dir in run_dirs:
    run = load_run(run_dir)
    row = dict(run["summary"])
    row["run_directory"] = str(run_dir)
    row["candidate_regions"] = len(run["regions"])
    selected = run["regions"].get("selected")
    row["selected_regions"] = int(selected.sum()) if selected is not None else 0
    run_summaries.append(row)

run_summary = pd.DataFrame(run_summaries)
run_summary.to_csv(ANALYSIS_OUTPUT / "run_summary.csv", index=False)
run_summary.to_json(
    ANALYSIS_OUTPUT / "run_summary.json",
    orient="records", indent=2,
)
run_summary

## 3. Combine the region tables

The ensemble table contains one row per modelled region. With `SELECTED_ONLY = True`, rejected candidate regions are excluded.

In [ ]:
regions = load_region_ensemble(RUNS_ROOT, selected_only=SELECTED_ONLY)
regions.to_csv(ANALYSIS_OUTPUT / "region_ensemble.csv", index=False)

print(f"Rows: {len(regions):,}")
print("Columns:", ", ".join(regions.columns))
regions.head()

In [ ]:
metrics = [
    column
    for column in [
        "area",
        "median_slope",
        "median_elevation",
        "local_relief",
        "slope_direction_length_new",
        "perpendicular_width_new",
        "mean_fos",
        "mean_critical_acceleration",
        "max_newmark_displacement",
    ]
    if column in regions.columns
]
regions[metrics].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

## 4. Distribution plots for every run

Each figure contains histograms and empirical cumulative distribution functions for area, slope, elevation, relief, length, and width.

In [ ]:
for run_dir in run_dirs:
    run = load_run(run_dir)
    plot_run(
        run,
        selected_only=SELECTED_ONLY,
        output_path=ANALYSIS_OUTPUT / f"{run_dir.name}.png",
    )
    plt.close()

print(f"Saved {len(run_dirs)} plot(s) to {ANALYSIS_OUTPUT.resolve()}")

## 5. Compare parameter groups

Choose one or more parameter columns present in the ensemble table. The example below compares landslide area by cohesion and soil-depth distribution, saving the compact summary as both CSV and readable JSON.

In [ ]:
requested_groups = ["cohesion_eff", "soil_distribution"]
group_columns = [column for column in requested_groups if column in regions.columns]

if group_columns and "area" in regions.columns:
    parameter_summary = (
        regions.groupby(group_columns, dropna=False)["area"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )
    parameter_summary.to_csv(
        ANALYSIS_OUTPUT / "area_by_parameters.csv", index=False
    )
    parameter_summary.to_json(
        ANALYSIS_OUTPUT / "area_by_parameters.json",
        orient="records", indent=2,
    )
    display(parameter_summary)
else:
    print("The requested grouping columns or area metric are not available.")

In [ ]:
if group_columns and "area" in regions.columns:
    figure, axis = plt.subplots(figsize=(10, 5), layout="constrained")
    regions.boxplot(column="area", by=group_columns, ax=axis, grid=False)
    axis.set_yscale("log")
    axis.set_ylabel("Area (m²)")
    axis.set_title("Selected landslide area by parameter group")
    figure.suptitle("")
    figure.savefig(ANALYSIS_OUTPUT / "area_by_parameters.png", dpi=200)

## 6. Inspect one run and its raster fields

Raster loading is opt-in. Zarr-backed runs are opened through xarray; `.npy` fallback files are memory-mapped. Select a run by changing `RUN_INDEX`.

In [ ]:
RUN_INDEX = -1  # newest discovered run
selected_run = load_run(run_dirs[RUN_INDEX], load_rasters=True)

print("Run:", selected_run["summary"].get("run_id", run_dirs[RUN_INDEX].name))
print("Execution mode:", selected_run["manifest"].get("runtime", {}).get("execution_mode"))
print(selected_run["rasters"])

In [ ]:
rasters = selected_run["rasters"]
available_fields = list(rasters.data_vars) if hasattr(rasters, "data_vars") else list(rasters)
available_fields

In [ ]:
preferred_field = "factor_of_safety"
RASTER_FIELD = preferred_field if preferred_field in available_fields else available_fields[0]

raster = rasters[RASTER_FIELD]
values = np.asarray(raster)

figure, axis = plt.subplots(figsize=(10, 7), layout="constrained")
image = axis.imshow(values, origin="lower", cmap="viridis")
axis.set_title(RASTER_FIELD)
axis.set_xlabel("Column")
axis.set_ylabel("Row")
figure.colorbar(image, ax=axis, shrink=0.8)
figure.savefig(ANALYSIS_OUTPUT / f"{run_dirs[RUN_INDEX].name}_{RASTER_FIELD}.png", dpi=200)

### Runout erosion, deposition, and net soil-depth change

These stable raster names are written by runout-enabled quick-start and full-grid CLI runs. The cell also reports gracefully when the selected run was created without runout.

In [ ]:
runout_fields = [
    ("runout_erosion", "Runout erosion (m)", "magma"),
    ("runout_deposition", "Runout deposition (m)", "viridis"),
    ("runout_soil_depth_change", "Net soil-depth change (m)", "RdBu"),
]
present_runout_fields = [item for item in runout_fields if item[0] in available_fields]

if not present_runout_fields:
    print("The selected run has no runout rasters. Select a runout-enabled run or use RUN_INDEX = -1 after running the quick start.")
else:
    figure, axes = plt.subplots(
        1, len(present_runout_fields), figsize=(5 * len(present_runout_fields), 4),
        layout="constrained", squeeze=False,
    )
    for axis, (field, title, cmap) in zip(axes.ravel(), present_runout_fields):
        values = np.asarray(rasters[field])
        limit = np.nanmax(np.abs(values)) if field.endswith("change") else None
        image = axis.imshow(
            values, origin="lower", cmap=cmap,
            vmin=-limit if limit else None, vmax=limit if limit else None,
        )
        axis.set_title(title)
        figure.colorbar(image, ax=axis, shrink=0.8)
    figure.savefig(ANALYSIS_OUTPUT / f"{run_dirs[RUN_INDEX].name}_runout.png", dpi=200)
    display(pd.Series({
        "source_nodes_above_displacement_threshold": selected_run["summary"].get("runout_source_node_count", 0),
        "traced_source_nodes": selected_run["summary"].get("runout_traced_source_node_count", 0),
        "excavated_source_nodes": selected_run["summary"].get("runout_excavated_source_node_count", 0),
        "terminated_multiflow_paths": selected_run["summary"].get("runout_terminated_path_count", 0),
        "mean_paths_per_moving_source": selected_run["summary"].get("runout_mean_paths_per_moving_source", 0.0),
        "max_paths_per_source": selected_run["summary"].get("runout_max_paths_per_source", 0),
        "source_proportion_sum_errors": selected_run["summary"].get("runout_source_proportion_error_count", 0),
        "changed_nodes": selected_run["summary"].get("runout_changed_node_count", 0),
        "total_erosion_node_m": selected_run["summary"].get("runout_total_erosion_node_m", 0.0),
        "total_deposition_node_m": selected_run["summary"].get("runout_total_deposition_node_m", 0.0),
        "mass_balance_error_node_m": selected_run["summary"].get(
            "runout_mass_balance_error_node_m",
            float(np.sum(rasters["runout_deposition"]) - np.sum(rasters["runout_erosion"])),
        ),
        "minimum_final_soil_depth_m": selected_run["summary"].get(
            "final_soil_depth_min_m", float(np.nanmin(rasters["soil_depth"])),
        ),
        "negative_final_soil_depth_nodes": selected_run["summary"].get(
            "negative_final_soil_depth_node_count",
            int(np.count_nonzero(np.asarray(rasters["soil_depth"]) < 0)),
        ),
    }, name="runout summary"))

### Selected initiation footprint versus runout footprint

The selected footprint is the set of candidate initiation nodes (`selected_labels > 0`). Within it, each node whose displacement exceeds the configured threshold independently starts its own multiflow runout tree and is excavated once if a valid moving path exists. Its material is divided among final path endpoints; intermediate routing nodes are not modified unless they are also independent source nodes. The table separates erosion, deposition, runout-only deposition, overlap, and their combined footprint.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

selected_footprint = (
    np.asarray(rasters["selected_footprint"], dtype=bool)
    if "selected_footprint" in available_fields
    else np.asarray(rasters["selected_labels"]) > 0
)
if {"runout_erosion", "runout_deposition"}.issubset(available_fields):
    erosion_footprint = np.asarray(rasters["runout_erosion"]) > 0
    deposition_footprint = np.asarray(rasters["runout_deposition"]) > 0
    runout_footprint = erosion_footprint | deposition_footprint
    net_change = (
        np.asarray(rasters["runout_soil_depth_change"])
        if "runout_soil_depth_change" in available_fields
        else np.asarray(rasters["runout_deposition"]) - np.asarray(rasters["runout_erosion"])
    )
    net_change_footprint = np.abs(net_change) > 0
else:
    erosion_footprint = np.zeros_like(selected_footprint)
    deposition_footprint = np.zeros_like(selected_footprint)
    runout_footprint = np.zeros_like(selected_footprint)
    net_change_footprint = np.zeros_like(selected_footprint)

runout_only = runout_footprint & ~selected_footprint
overlap = runout_footprint & selected_footprint
combined = runout_footprint | selected_footprint
node_area_m2 = (
    selected_run["manifest"]["grid"]["dx"]
    * selected_run["manifest"]["grid"]["dy"]
)
footprints = {
    "selected initiation": selected_footprint,
    "runout erosion sources": erosion_footprint,
    "runout deposition endpoints": deposition_footprint,
    "runout affected (erosion or deposition)": runout_footprint,
    "runout only": runout_only,
    "selected/runout overlap": overlap,
    "combined affected": combined,
    "runout net soil-depth change": net_change_footprint,
}
footprint_summary = pd.DataFrame([
    {"footprint": name, "node_count": int(mask.sum()),
     "area_m2": float(mask.sum() * node_area_m2)}
    for name, mask in footprints.items()
])
footprint_summary.to_csv(ANALYSIS_OUTPUT / "footprint_summary.csv", index=False)
footprint_summary.to_json(
    ANALYSIS_OUTPUT / "footprint_summary.json", orient="records", indent=2
)
display(footprint_summary)

footprint_class = np.zeros(selected_footprint.shape, dtype=np.uint8)
footprint_class[selected_footprint & ~runout_footprint] = 1
footprint_class[overlap] = 2
footprint_class[runout_only] = 3
colors = ["#f2f2f2", "#d73027", "#762a83", "#4575b4"]
labels = ["Unaffected", "Selected initiation only", "Selected + runout", "Runout only"]
figure, axis = plt.subplots(figsize=(10, 7), layout="constrained")
axis.imshow(footprint_class, origin="lower", cmap=ListedColormap(colors), vmin=0, vmax=3)
axis.set_title("Selected initiation and runout footprints")
axis.legend(
    handles=[Patch(facecolor=color, label=label) for color, label in zip(colors, labels)],
    loc="upper right",
)
figure.savefig(
    ANALYSIS_OUTPUT / f"{run_dirs[RUN_INDEX].name}_selected_vs_runout.png", dpi=200
)

### Large-DEM note

`np.asarray(raster)` materialises the selected raster in memory. This is normally reasonable for one field, but avoid converting several very large rasters at once. Work with slices such as `raster[::4, ::4]` for a quick-look plot when memory is limited.

## 7. Example custom filtering

The region table is an ordinary pandas DataFrame, so scientific subsets can be expressed directly. Adjust the thresholds to suit the DEM resolution and research question.

In [ ]:
filtered = regions.copy()
if "area" in filtered.columns:
    filtered = filtered[filtered["area"] >= 1_000]
if "median_slope" in filtered.columns:
    filtered = filtered[filtered["median_slope"] >= 20]

filtered.to_csv(ANALYSIS_OUTPUT / "filtered_regions.csv", index=False)
print(f"Retained {len(filtered):,} of {len(regions):,} regions")
filtered.head()